In [1]:
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.preprocessing import compute_proj_ecg
from mne_connectivity import envelope_correlation
import pandas as pd
from scipy.fft import fft,ifft
from sklearn.decomposition import PCA
from sklearn import svm
import pywt
from scipy.signal import stft
from scipy.signal import ShortTimeFFT
import os
from scipy.stats import kurtosis, skew, zscore
from scipy import fft
import emd
import antropy
import scipy
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import cross_validate, cross_val_score, cross_val_predict
#%matplotlib qt

In [2]:
import ipynb
from ipynb.fs.full.utils import *

In [3]:
def statistical_features(x):
    x_features= np.zeros((x.shape[0],4))
    x_features[:,0]= np.mean(x, axis=1)
    x_features[:,1]= np.var(x, axis=1)
    x_features[:,2]= skew(x, axis=1)
    x_features[:,3]= kurtosis(x, axis=1)
    return x_features

In [4]:
def hjorth_features(x):
    x_hjorth= np.zeros((x.shape[0],2))
    params= antropy.hjorth_params(x)
    x_hjorth[:,0]= params[0]
    x_hjorth[:,1]= params[1]
    return x_hjorth

In [5]:
def bandpower(x, fs, fmin, fmax):
    f, Pxx = scipy.signal.welch(x, fs=fs)
    #print(f.shape)
    #print(Pxx.shape)
    ind_min = np.argmax(f > fmin) - 1
    #print(ind_min)
    ind_max = np.argmax(f > fmax) - 1
    #print(ind_max)
    return np.trapz(Pxx[:,ind_min: ind_max], f[ind_min: ind_max])

In [6]:
def bandpower_extraction(x):
    x_power= np.zeros((x.shape[0], 5))
    x_power[:,0]= bandpower(x, 200, 0.5, 4)
    x_power[:,1]= bandpower(x, 200, 4, 8)
    x_power[:,2]= bandpower(x, 200, 8, 12)
    x_power[:,3]= bandpower(x, 200, 12, 30)
    x_power[:,4]= bandpower(x, 200, 30, 120)
    return x_power

In [7]:
def feature_extraction(x):
    x_features= statistical_features(x)
    x_features= np.append(x_features, hjorth_features(x), axis=1)
    x_features= np.append(x_features, bandpower_extraction(x), axis=1)
    return x_features

In [8]:
def imf_features(x, num_imf=5):
    num_features=11
    x_imf=np.zeros((x.shape[0],num_features*num_imf))
    for i,x0 in enumerate(x):
        imf = emd.sift.sift(x0)
        #print(imf.shape)
        if imf.shape[1]<num_imf:
            continue
        imf= imf.T[:num_imf]
        imf_features= feature_extraction(imf)
        imf_features=imf_features.flatten()
        #print(i)
        x_imf[i]=imf_features
    return x_imf

In [1]:
def cwt_features(x, wavelet='morl', freqs=np.arange(1,100.5,10), fs=200):
    coef, freqs= CWT_transform(x, wavelet, freqs, fs)
    num_scales= len(freqs)
    num_features=11
    cwt_features= np.zeros((x.shape[0],num_features,num_scales))
    #print(cwt_features.shape)
    for i in range(num_scales):
        #feat=feature_extraction(coef[:,i])
        #print(feat.shape)
        #print(cwt_features[:,:,i].shape)
        cwt_features[:,:,i]= feature_extraction(coef[:,i])
    return cwt_features.reshape((cwt_features.shape[0], num_features*num_scales))

NameError: name 'np' is not defined

In [10]:
def get_features_full(x, imf=True, cwt=True, pca=True):
    x_features= feature_extraction(x)
    print(x_features.shape)
    if imf:
        imf_feat=imf_features(x)
        print(imf_feat.shape)
        x_features= np.append(x_features, imf_feat, axis=1)
    if cwt:
        cwt_feat=cwt_features(x)
        print(cwt_feat.shape)
        x_features= np.append(x_features, cwt_feat, axis=1)
    if pca:
        pca_feat= PCA_transform(x)
        print(pca_feat.shape)
        x_features= np.append(x_features, pca_feat, axis=1)
    print(np.argwhere(np.isnan(x_features)))
    x_features=np.nan_to_num(x_features)
    return x_features

In [11]:
def normalize(x_train, x_test):
    mean= np.mean(x_train, axis=0)
    std= np.std(x_train, axis=0)
    x_train= zscore(x_train, axis=0) 
    x_test= np.divide((x_test- mean), std)
    return x_train, x_test, mean, std

In [12]:
def run_svm(x_train, x_test, y_train, y_test, kernel='rbf', degree=3, C=1, gamma='scale', coef0=0,
            class_weight=None, return_scores=False):
    print('Training data: ' + str(len(x_train)))
    print('Training data by class: ')
    print('Ictal: '+ str(len(np.where(y_train==0)[0])))
    print('Pre-ictal: '+ str(len(np.where(y_train==1)[0])))
    print('Non-ictal: '+ str(len(np.where(y_train==2)[0])))
    print('Testing data: ' + str(len(x_test)))
    print('Testing data by class: ')
    print('Ictal: '+ str(len(np.where(y_test==0)[0])))
    print('Pre-ictal: '+ str(len(np.where(y_test==1)[0])))
    print('Non-ictal: '+ str(len(np.where(y_test==2)[0])))
    #x_train, x_test, mean, std= normalize(x_train, x_test)
    scaler= StandardScaler()
    x_train= scaler.fit_transform(x_train)
    x_test= scaler.transform(x_test)
    #print(np.argwhere(np.isnan(x_train)))
    #x_train= np.nan_to_num(x_train)
    #x_test= np.nan_to_num(x_test)
    svm_= svm.SVC(kernel=kernel, degree=degree, C=C, gamma=gamma, coef0=coef0, class_weight=class_weight)
    svm_.fit(x_train, y_train)
    print('Training accuracy:')
    train_acc= svm_.score(x_train, y_train)
    print(train_acc)
    y_train_predict= svm_.predict(x_train)
    train_f1_score= f1_score(y_train, y_train_predict, average=None)
    print('Training accuracy by class:')
    print(train_f1_score)
    print('Testing accuracy:')
    test_acc= svm_.score(x_test, y_test)
    print(test_acc)
    y_predict=svm_.predict(x_test)
    test_f1_score= f1_score(y_test, y_predict, average=None)
    print('Testing accuracy by class:')
    print(test_f1_score)
    if return_scores:
        return svm_, scaler, train_acc, test_acc, train_f1_score, test_f1_score
    else:
        return svm_, scaler

In [13]:
def run_pipeline(x_train, x_test, y_train, y_test, scaler=StandardScaler(), model=svm.SVC(), return_scores=False):
    print('Training data: ' + str(len(x_train)))
    print('Training data by class: ')
    print('Ictal: '+ str(len(np.where(y_train==0)[0])))
    print('Pre-ictal: '+ str(len(np.where(y_train==1)[0])))
    print('Non-ictal: '+ str(len(np.where(y_train==2)[0])))
    print('Testing data: ' + str(len(x_test)))
    print('Testing data by class: ')
    print('Ictal: '+ str(len(np.where(y_test==0)[0])))
    print('Pre-ictal: '+ str(len(np.where(y_test==1)[0])))
    print('Non-ictal: '+ str(len(np.where(y_test==2)[0])))
    pipe= make_pipeline(scaler, model)
    pipe.fit(x_train,y_train)
    print('Training accuracy:')
    train_acc= pipe.score(x_train, y_train)
    print(train_acc)
    y_train_predict= pipe.predict(x_train)
    train_f1_score= f1_score(y_train, y_train_predict, average=None)
    print('Training accuracy by class:')
    print(train_f1_score)
    print('Testing accuracy:')
    test_acc= pipe.score(x_test, y_test)
    print(test_acc)
    y_predict=pipe.predict(x_test)
    test_f1_score= f1_score(y_test, y_predict, average=None)
    print('Testing accuracy by class:')
    print(test_f1_score)
    if return_scores:
        return pipe, train_acc, test_acc, train_f1_score, test_f1_score
    else:
        return pipe

In [14]:
def run_svm_cv(x, y, metadata, split, kernel='rbf', degree=3, C=1, gamma='scale', coef0=0, class_weight=None):
    train_acc= np.array([])
    test_acc= np.array([])
    train_f1_score= np.array([])
    #print(train_f1_score.size)
    test_f1_score= np.array([])
    for i, (train, test) in enumerate(split):
        print('Fold: '+ str(i))
        x_train, x_test= x[train], x[test]
        y_train, y_test= y[train], y[test]
        meta_train, meta_test= metadata.loc[train], metadata.loc[test]
        svm_, scaler, train_acc_0, test_acc_0, train_f1_score_0, test_f1_score_0=run_svm(
            x_train, x_test, y_train, y_test, kernel=kernel, degree=degree, C=C, gamma=gamma, coef0=coef0, class_weight=class_weight,
            return_scores=True)
        train_f1_score_0=np.reshape(train_f1_score_0, (1,3))
        test_f1_score_0=np.reshape(test_f1_score_0, (1,3))
        if train_f1_score.size==0:
            train_acc= np.array([train_acc_0])
            test_acc= np.array([test_acc_0])
            train_f1_score=train_f1_score_0
            test_f1_score=test_f1_score_0
        else:
            train_acc=np.append(train_acc, train_acc_0)
            test_acc=np.append(test_acc,test_acc_0)
            train_f1_score=np.append(train_f1_score, train_f1_score_0, axis=0)
            test_f1_score=np.append(test_f1_score, test_f1_score_0, axis=0)
    #print(train_acc.shape)
    #print(test_acc.shape)
    #print(train_f1_score.shape)
    #print(test_f1_score.shape)
    train_avg= np.mean(train_acc)
    test_avg= np.mean(test_acc)
    train_f1_avg= np.mean(train_f1_score, axis=0)
    test_f1_avg= np.mean(test_f1_score, axis=0)
    print('Average training accuracy: '+ str(train_avg))
    print('By class: ')
    print(train_f1_avg)
    print('Average testing accuracy: '+ str(test_avg))
    print('By class: ')
    print(test_f1_avg)
    return svm_, scaler

In [15]:
def run_svm_cv(x, y, metadata, split, kernel='rbf', degree=3, C=1, gamma='scale', coef0=0, class_weight=None):
    train_acc= np.array([])
    test_acc= np.array([])
    train_f1_score= np.array([])
    #print(train_f1_score.size)
    test_f1_score= np.array([])
    for i, (train, test) in enumerate(split):
        print('Fold: '+ str(i))
        x_train, x_test= x[train], x[test]
        y_train, y_test= y[train], y[test]
        meta_train, meta_test= metadata.loc[train], metadata.loc[test]
        svm_, scaler, train_acc_0, test_acc_0, train_f1_score_0, test_f1_score_0=run_svm(
            x_train, x_test, y_train, y_test, kernel=kernel, degree=degree, C=C, gamma=gamma, coef0=coef0, class_weight=class_weight,
            return_scores=True)
        train_f1_score_0=np.reshape(train_f1_score_0, (1,3))
        test_f1_score_0=np.reshape(test_f1_score_0, (1,3))
        if train_f1_score.size==0:
            train_acc= np.array([train_acc_0])
            test_acc= np.array([test_acc_0])
            train_f1_score=train_f1_score_0
            test_f1_score=test_f1_score_0
        else:
            train_acc=np.append(train_acc, train_acc_0)
            test_acc=np.append(test_acc,test_acc_0)
            train_f1_score=np.append(train_f1_score, train_f1_score_0, axis=0)
            test_f1_score=np.append(test_f1_score, test_f1_score_0, axis=0)
    #print(train_acc.shape)
    #print(test_acc.shape)
    #print(train_f1_score.shape)
    #print(test_f1_score.shape)
    train_avg= np.mean(train_acc)
    test_avg= np.mean(test_acc)
    train_f1_avg= np.mean(train_f1_score, axis=0)
    test_f1_avg= np.mean(test_f1_score, axis=0)
    print('Average training accuracy: '+ str(train_avg))
    print('By class: ')
    print(train_f1_avg)
    print('Average testing accuracy: '+ str(test_avg))
    print('By class: ')
    print(test_f1_avg)
    return svm_, scaler

In [17]:
def run_pipeline_cv_2(x, y, metadata, cv=StratifiedGroupKFold(n_splits=5), scaler=StandardScaler(), model=svm.SVC()):
    num_classes= np.max(y)+1
    pipe= make_pipeline(scaler, model)
    scores= cross_validate(pipe, x, y, groups=metadata['File'], cv=cv,
                        return_train_score=True, return_estimator=True, return_indices=True)
    train_acc=scores['train_score']
    print('Training accuracy: ')
    print(train_acc)
    train_f1= np.zeros((cv.n_splits, num_classes))
    test_f1= np.zeros((cv.n_splits, num_classes))
    print('By class: ')
    for i,model in enumerate(scores['estimator']):
        train, test= scores['indices']['train'][i], scores['indices']['test'][i]
        x_train, x_test, y_train, y_test= x[train], x[test], y[train], y[test]
        y_train_predict= model.predict(x_train)
        train_f1[i]= f1_score(y_train, y_train_predict, average=None)
        y_test_predict= model.predict(x_test)
        test_f1[i]= f1_score(y_test, y_test_predict, average=None)
    print(train_f1)
    test_acc=scores['test_score']
    print('Testing accuracy: ')
    print(test_acc)
    print('By class: ')
    print(test_f1)
    train_avg= np.mean(train_acc)
    test_avg= np.mean(test_acc)
    train_f1_avg= np.mean(train_f1, axis=0)
    test_f1_avg= np.mean(test_f1, axis=0)
    print('Average training accuracy: '+ str(train_avg))
    print('By class: ')
    print(train_f1_avg)
    print('Average testing accuracy: '+ str(test_avg))
    print('By class: ')
    print(test_f1_avg)
    return scores